In [ ]:
%run ../../commonsetup/Setup-Common

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_route_delay_summary_v AS
    SELECT
      gtfs_mode,
      route_id,
      COALESCE(route_short_name, 'Unscheduled/Unresolved') AS route_short_name,
      route_long_name,
      date_trunc('HOUR', poll_timestamp) AS poll_hour,
      COUNT(*) AS reading_count,
      ROUND(AVG(arrival_delay_seconds) / 60.0, 1) AS avg_arrival_delay_minutes,
      ROUND(MAX(arrival_delay_seconds) / 60.0, 1) AS max_arrival_delay_minutes,
      ROUND(AVG(departure_delay_seconds) / 60.0, 1) AS avg_departure_delay_minutes
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.gold_trip_delays_enriched
    WHERE arrival_delay_seconds IS NOT NULL OR departure_delay_seconds IS NOT NULL
    GROUP BY ALL
    ORDER BY poll_hour DESC, avg_arrival_delay_minutes DESC
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {CATALOG_NAME}.{SCHEMA_NAME}.gold_vehicle_positions_current_v AS
    SELECT
      vp.gtfs_mode,
      vp.vehicle_id,
      vp.vehicle_label,
      vp.trip_id,
      COALESCE(vp.route_id, t.route_id) AS route_id,
      COALESCE(r.route_short_name, 'Unscheduled/Unresolved') AS route_short_name,
      r.route_long_name,
      t.trip_headsign, t.direction_id,
      vp.schedule_relationship, vp.latitude, vp.longitude, vp.position_timestamp,
      vp.stop_id, s.stop_name, vp.congestion_level,
      COALESCE(vp.occupancy_status, 'Not Reported') AS occupancy_status,
      vp.poll_timestamp
    FROM {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_vehicle_positions_current vp
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_trips  t ON vp.gtfs_mode = t.gtfs_mode AND vp.trip_id = t.trip_id
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_routes r ON vp.gtfs_mode = r.gtfs_mode AND COALESCE(vp.route_id, t.route_id) = r.route_id
    LEFT JOIN {CATALOG_NAME}.{SCHEMA_NAME}.silver_gtfs_stops  s ON vp.gtfs_mode = s.gtfs_mode AND vp.stop_id = s.stop_id
""")